<a href="https://colab.research.google.com/github/MElsdk-lab/Biochar_forest_estimation/blob/main/Notebook_1_GEE_Forest_Area_Computation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# NOTEBOOK 1 — GEE Forest Area Computation
# University of Pittsburgh | Biochar Feedstock Methodology
# ============================================================

In [2]:
# ── CELL 1: Install Libraries ─────────────────────────────────────────────────
!pip install -q earthengine-api geemap

In [3]:
# ── CELL 2: clone repo if not already cloned ─────────────────────
import os
import getpass
import subprocess


if not os.path.exists('/content/Biochar_forest_estimation'):
    PAT = getpass.getpass('Enter PAT: ')
    !git config --global user.email "sdkmajd@gmail.com"
    !git config --global user.name "MElsdk-lab"
    subprocess.run(
        f'git clone https://{PAT}@github.com/MElsdk-lab/Biochar_forest_estimation.git',
        shell=True
    )

%cd /content/Biochar_forest_estimation/
!git fetch origin
!git reset --hard origin/main
print('✅ Ready')

/content/Biochar_forest_estimation
HEAD is now at 39d34c2 remove import ee duplicate
✅ Ready


In [4]:
# ── CELL 3: import libraries and data from data_config ─────────────────────

import ee
import geemap
import pandas as pd
from data_config import FAO_LSIB_REGION, us_state_names, get_all_countries, build_country_lookup, country_thresholds, state_thresholds

print('✅ Libraries imported')
print('✅ Data config loaded')

✅ Libraries imported
✅ Data config loaded


In [5]:
# ── CELL 4: Authenticate and initialize Earth Engine & Load Datasets  ─────────────────────
try:
    ee.Initialize(project='spry-blade-487218-n0')
except Exception as e:
    ee.Authenticate()
    ee.Initialize(project='spry-blade-487218-n0')  # ← add project here too

Hansen_GFC_2024      = ee.Image("UMD/hansen/global_forest_change_2024_v1_12")
GLC_FSC30D_annual    = ee.ImageCollection('projects/sat-io/open-datasets/GLC-FCS30D/annual')
GLC_FSC30D_five_year = ee.ImageCollection('projects/sat-io/open-datasets/GLC-FCS30D/five-years-map')

print('✅ Hansen GFC 2024 loaded')
print('✅ GLC FCS30D annual loaded')
print('✅ GLC FCS30D five-year loaded')


✅ Hansen GFC 2024 loaded
✅ GLC FCS30D annual loaded
✅ GLC FCS30D five-year loaded


In [6]:
# ── CELL 5: processing datasets ───────────────────────────────────────

# Select & Mask Hansen Bands
treecover2000 = Hansen_GFC_2024.select('treecover2000')
datamask      = Hansen_GFC_2024.select('datamask')

treecover2000_masked = treecover2000.updateMask(treecover2000.gt(0)) \
                                    .updateMask(datamask.eq(1))

print('✅ treecover2000 masked')

#Load GLC FCS30D Year 2000

glc_2000        = GLC_FSC30D_annual.mosaic().select('b1')
glc_2000_forest = glc_2000.gte(51).And(glc_2000.lte(92)).selfMask()

print('✅ GLC FCS30D 2000 loaded')
print('✅ GLC forest mask created (classes 51-92)')

# Forest Class Definitions
forestClasses = [
    {'code': 51, 'color': '4c7300', 'name': '51 Open evergreen broadleaved'},
    {'code': 52, 'color': '006400', 'name': '52 Closed evergreen broadleaved'},
    {'code': 61, 'color': 'a8c800', 'name': '61 Open deciduous broadleaved'},
    {'code': 62, 'color': '00a000', 'name': '62 Closed deciduous broadleaved'},
    {'code': 71, 'color': '005000', 'name': '71 Open evergreen needleleaved'},
    {'code': 72, 'color': '003c00', 'name': '72 Closed evergreen needleleaved'},
    {'code': 81, 'color': '286400', 'name': '81 Open deciduous needleleaved'},
    {'code': 82, 'color': '285000', 'name': '82 Closed deciduous needleleaved'},
    {'code': 91, 'color': 'a0b432', 'name': '91 Open mixed forest'},
    {'code': 92, 'color': '788200', 'name': '92 Closed mixed forest'},
]

print(f'✅ {len(forestClasses)} forest classes defined')

✅ treecover2000 masked
✅ GLC FCS30D 2000 loaded
✅ GLC forest mask created (classes 51-92)
✅ 10 forest classes defined


In [12]:
# ── CELL 6: GEE Functions  — Countries ────────────────────────────────────────
def prepare_countries_forest_collection(selected_regions, thresholds):
    """
    Prepare a GEE FeatureCollection with forest area per country
    for a given canopy cover threshold.
    Returns a GEE object (not yet computed).
    """
    all_countries = get_all_countries(selected_regions)
    lsib_fao = ee.FeatureCollection('USDOS/LSIB_SIMPLE/2017') \
                 .filter(ee.Filter.inList('country_na', all_countries))

    country_results =[]

    for threshold in thresholds:
      forest_mask = treecover2000_masked.gte(threshold) \
                                      .selfMask() \
                                      .updateMask(datamask.eq(1))

      area_image = forest_mask.multiply(ee.Image.pixelArea().divide(1e10))

      region_areas = area_image.reduceRegions(
        collection=lsib_fao,
        reducer=ee.Reducer.sum(),
        scale=30,
      ).getInfo()


      for feature in region_areas['features']:

        country_results.append({
            'country': feature['properties']['country_na'],
            'threshold': threshold,
            'area': feature['properties'].get('sum',0)
        })

    return pd.DataFrame(country_results)


print('✅ prepare_forest_collection() defined')
print('✅ export_forest_area() defined')

✅ prepare_forest_collection() defined
✅ export_forest_area() defined


In [8]:
# ── CELL 7: GEE Functions — US States ───────────────────────────────────────
tiger_states = ee.FeatureCollection('TIGER/2018/States') \
                 .filter(ee.Filter.inList('NAME', us_state_names))

selected_states = tiger_states

def prepare_states_forest_collection(selected_states, thresholds):
    """
    Prepare a GEE FeatureCollection with forest area per US state
    for a given canopy cover threshold.
    Returns a GEE object (not yet computed).
    """
    states_results = []
    for threshold in thresholds:
      forest_mask = treecover2000_masked.gte(threshold) \
                                      .selfMask() \
                                      .updateMask(datamask.eq(1))

      area_image = forest_mask.multiply(ee.Image.pixelArea().divide(1e10))

      states_areas = area_image.reduceRegions(
        collection=selected_states,
        reducer=ee.Reducer.sum(),
        scale=30,
        maxPixelsPerRegion=1e13
      ).getInfo()

      for feature in states_areas['features']:
        states_results.append({
            'state': feature['properties']['NAME'],
            'threshold': threshold,
            'area': feature['properties'].get('sum',0)
        })

    return pd.DataFrame(states_results)




print('✅ prepare_states_forest_collection() defined')
print('✅ export_states_forest_area() defined')
print(f'✅ {len(us_state_names)} US states defined')

✅ prepare_states_forest_collection() defined
✅ export_states_forest_area() defined
✅ 50 US states defined


In [15]:
DATA_folder = '/content/Biochar_forest_estimation/data/'

#countries
print('── Computing country forest area ──')
df_countries = prepare_countries_forest_collection(FAO_LSIB_REGION, country_thresholds)
df_countries.to_csv(DATA_folder + 'country_total_forest_area.csv', index=False)
print(f'✅ Countries done — {len(df_countries)} rows')

#states
print('\n── Computing state forest area ──')
df_states = prepare_states_forest_collection(tiger_states, state_thresholds)
df_states.to_csv(DATA_folder + 'state_total_forest_area.csv', index=False)
print(f'✅ States done — {len(df_states)} rows')

── Computing country forest area ──
✅ Countries done — 1100 rows

── Computing state forest area ──
✅ States done — 250 rows


In [16]:
# ── CELL 8: Push results to GitHub ────────────────────────────────
import subprocess

# ask for PAT only if not already defined
if 'PAT' not in dir():
    import getpass
    PAT = getpass.getpass('Enter PAT: ')

%cd /content/Biochar_forest_estimation/
!git add .
!git commit -m "update results"

result = subprocess.run(
    f'git push https://{PAT}@github.com/MElsdk-lab/Biochar_forest_estimation.git main',
    shell=True,
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print('✅ Pushed to GitHub')
else:
    print('❌ Push failed')
    print(result.stderr)

/content/Biochar_forest_estimation
[main 0508efa] update results
 2 files changed, 1279 insertions(+), 1154 deletions(-)
✅ Pushed to GitHub
